# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/naca10795-ops/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: 1 row = 1 unique customer profile (CustomerID) in the mall customer slice.

Time Window: Mid-panel observation period (e.g., 2026-03-01 to 2026-03-31).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load/Inspect the slice schema
df = pd.DataFrame({
    'CustomerID': range(1, 201),
    'Age': [19, 21, 20, 23, 31] * 40,
    'Annual_Income_k': [15, 15, 16, 16, 17] * 40,
    'Spending_Score': [39, 81, 6, 77, 40] * 40,
    'observation_month': ['2026-03'] * 200,
    'is_active': [True] * 190 + [False] * 10
})

print(f"Row count: {len(df)}")
print(f"Unique Customers: {df['CustomerID'].nunique()}")

Row count: 200
Unique Customers: 200


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: Age, Annual_Income_k, Spending_Score, Income_to_Age_Ratio.

Label / Proxy: is_high_value_target (Annual_Income_k > 70 AND Spending_Score > 60).

Context: observation_month, CustomerID.

Excluded: Future transaction records beyond 2026-03 (to avoid temporal data leakage) and raw internal system keys.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['Income_to_Age_Ratio'] = df['Annual_Income_k'] / df['Age']
df['is_high_value_target'] = ((df['Annual_Income_k'] > 70) & (df['Spending_Score'] > 60)).astype(int)
print("Field buckets defined. Target distribution:")
print(df['is_high_value_target'].value_counts())

Field buckets defined. Target distribution:
is_high_value_target
0    200
Name: count, dtype: int64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Checks:

    Grain check (confirming no duplicate CustomerIDs).

    Date window verification.

    Availability check using IS TRUE filter.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# 1. Grain check (Duplicates)
dup_check = duckdb.query("SELECT CustomerID, COUNT(*) FROM df GROUP BY CustomerID HAVING COUNT(*) > 1").df()
print("Duplicate Grain Check (Should be empty):", len(dup_check))

# 2. Date Window check
date_check = duckdb.query("SELECT MIN(observation_month), MAX(observation_month), COUNT(*) FROM df").df()
print("\nDate Window & Total Counts:\n", date_check)

# 3. Availability Check (IS TRUE)
avail_check = duckdb.query("SELECT COUNT(*) FROM df WHERE is_active IS TRUE").df()
print("\nActive Rows (is_active IS TRUE):\n", avail_check)

Duplicate Grain Check (Should be empty): 0

Date Window & Total Counts:
   min(observation_month) max(observation_month)  count_star()
0                2026-03                2026-03           200

Active Rows (is_active IS TRUE):
    count_star()
0           190


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limitation: The current mid-panel dataset snapshot reflects static monthly demographic distributions. It does not capture intra-month daily shopping cadence changes or real-time event logs prior to the monthly batch sync.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature Leakage Experiment (The Trap)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = ['Age', 'Annual_Income_k', 'Spending_Score', 'Income_to_Age_Ratio']
X = df[features]
y = df['is_high_value_target']

clf = RandomForestClassifier(random_state=42)
clf.fit(X, y)
print(f"Honest Feature Baseline Accuracy: {accuracy_score(y, clf.predict(X)):.4f}")

Honest Feature Baseline Accuracy: 1.0000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decsion-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.